# VAR Spillover Analysis: United States as Target

15-variable VAR on NS factors (5 countries). Cholesky ordering: EU -> CA -> BR -> UK -> US

Target: **US** (US Treasury Curve)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.var_analysis import (
    build_var_order, prepare_var_data, adf_tests,
    select_var_lag_detailed, estimate_var, compute_irfs,
    compute_fevd, fevd_summary, granger_causality_tests,
    var_diagnostics, FACTOR_NAMES,
)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)

TARGET = 'US'
ALL_COUNTRIES = ['US', 'EU', 'CA', 'BR', 'UK']
OTHERS = [c for c in ALL_COUNTRIES if c != TARGET]

## 1. Load and Prepare Data

In [ ]:
factors_raw = pd.read_csv('../data/factors/ns_factors_5c.csv', index_col=0, parse_dates=True)

factors_dict = {}
for cc in ALL_COUNTRIES:
    cols = [c for c in factors_raw.columns if c.startswith(f'{cc}_')]
    factors_dict[cc] = factors_raw[cols].rename(columns=lambda x: x.replace(f'{cc}_', ''))

var_data = prepare_var_data(factors_dict, target=TARGET, countries=ALL_COUNTRIES)
print(f'VAR data: {var_data.shape}')
print(f'Date range: {var_data.index.min()} to {var_data.index.max()}')
print(f'Columns: {list(var_data.columns)}')

## 2. Stationarity Tests

In [ ]:
adf_levels = adf_tests(var_data)
print('ADF Tests (Levels):')
print(adf_levels[['ADF_stat', 'p_value', 'stationary_5%']].to_string())

## 3. Lag Selection & Estimation

In [ ]:
lag_result = select_var_lag_detailed(var_data, max_lags=4)
print('Optimal lag by criterion:')
for criterion, lag in lag_result['recommended'].items():
    print(f'  {criterion}: {lag}')

LAGS = lag_result['recommended']['BIC']
print(f'\nUsing lag: {LAGS}')

var_results = estimate_var(var_data, lags=LAGS)
print(f'VAR({LAGS}) estimated. Obs: {var_results.nobs}')

## 4. Diagnostics

In [ ]:
diag = var_diagnostics(var_results)
print('Durbin-Watson:')
for var, dw in diag['durbin_watson'].items():
    if var.startswith(TARGET):
        print(f'  {var}: {dw:.3f}')
print(f'\nPortmanteau: stat={diag["portmanteau"]["statistic"]:.1f}, '
      f'p={diag["portmanteau"]["p_value"]:.4f}')

## 5. Impulse Response Functions

Response of US factors to all foreign shocks.

In [ ]:
HORIZON = 40
irf = var_results.irf(HORIZON)
irf_lower, irf_upper = irf.errband_mc(orth=True, repl=500, seed=42)

var_names = list(var_results.names)
target_responses = [f'{TARGET}_{f}' for f in FACTOR_NAMES]
foreign_shocks = [f'{c}_{f}' for c in OTHERS for f in FACTOR_NAMES]

n_foreign = len(OTHERS)
fig, axes = plt.subplots(3, n_foreign * 3, figsize=(5 * n_foreign, 8), sharex=True)

for i, response in enumerate(target_responses):
    for j, impulse in enumerate(foreign_shocks):
        ax = axes[i, j]
        imp_idx = var_names.index(impulse)
        resp_idx = var_names.index(response)
        irf_vals = irf.orth_irfs[:, resp_idx, imp_idx]
        ax.plot(range(HORIZON + 1), irf_vals, 'b-', lw=1.2)
        ax.axhline(0, color='grey', lw=0.5, ls='--')
        ax.fill_between(range(HORIZON + 1),
                        irf_lower[:, resp_idx, imp_idx],
                        irf_upper[:, resp_idx, imp_idx],
                        alpha=0.15, color='blue')
        if i == 0:
            ax.set_title(impulse.replace('_', ' '), fontsize=7)
        if j == 0:
            ax.set_ylabel(response.replace('_', ' '), fontsize=8)
        ax.tick_params(labelsize=5)

plt.suptitle('IRFs: US Responses to Foreign Shocks (95% CI)', fontsize=11)
plt.tight_layout()
plt.savefig('fig_irf.png', dpi=200, bbox_inches='tight')
plt.show()

## 6. Forecast Error Variance Decomposition

In [ ]:
fevd = compute_fevd(var_results, periods=52)
horizons = [1, 4, 12, 26, 52]

for response in target_responses:
    print(f'\nFEVD for {response}:')
    summary = fevd_summary(fevd, response, horizons=horizons, var_names=var_names)
    # Aggregate by country
    agg = {}
    for cc in ALL_COUNTRIES:
        cc_cols = [c for c in summary.columns if c.startswith(f'{cc}_')]
        agg[cc] = summary[cc_cols].sum(axis=1)
    print(pd.DataFrame(agg).round(1).to_string())

In [ ]:
# FEVD bar charts
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

for idx, response in enumerate(target_responses):
    ax = axes[idx]
    summary = fevd_summary(fevd, response, horizons=horizons, var_names=var_names)
    agg = pd.DataFrame()
    for i, cc in enumerate(ALL_COUNTRIES):
        cc_cols = [c for c in summary.columns if c.startswith(f'{cc}_')]
        label = f'{cc} (own)' if cc == TARGET else cc
        agg[label] = summary[cc_cols].sum(axis=1)
    agg.plot(kind='bar', stacked=True, ax=ax, color=colors)
    ax.set_title(response.replace('_', ' '))
    ax.set_xlabel('Horizon (weeks)')
    ax.set_ylabel('% Variance')
    ax.set_ylim(0, 105)
    ax.legend(loc='lower right', fontsize=6)
    ax.set_xticklabels(horizons, rotation=0)

plt.suptitle('FEVD — US Factors', fontsize=12)
plt.tight_layout()
plt.savefig('fig_fevd.png', dpi=200, bbox_inches='tight')
plt.show()

## 7. Granger Causality

In [ ]:
for cc in OTHERS:
    causing = [f'{cc}_{f}' for f in FACTOR_NAMES]
    gc = granger_causality_tests(var_results, causing=causing, target=TARGET)
    print(f'\n{cc} -> {TARGET}:')
    print(gc[['caused', 'F_stat', 'p_value', 'significant']].to_string(index=False))

## 8. Factor Time Series

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
for i, factor in enumerate(FACTOR_NAMES):
    ax = axes[i]
    for cc in ALL_COUNTRIES:
        ax.plot(factors_dict[cc].index, factors_dict[cc][factor],
                label=cc, alpha=0.8, lw=0.7)
    ax.set_ylabel(factor)
    ax.legend(loc='upper right', fontsize=7)
plt.suptitle('NS Factors — All Countries (target: US)', fontsize=11)
plt.tight_layout()
plt.savefig('fig_factors.png', dpi=200, bbox_inches='tight')
plt.show()